In [1]:
import os
import sys
sys.path.append(os.path.abspath('..'))
import logging
import warnings
import numpy as np
import re
import pandas as pd
import itables
from itables import show
from pathlib import Path
from dotenv import load_dotenv
from src.build_dataset import get_file_pairs, merge_qa_data, detect_exercise_type, find_answer_index, apply_reference_tag, compare
from src.evaluation import process_matching_row, apply_matching_processing

# --- Setup Warnings ---
warnings.filterwarnings('ignore', category=SyntaxWarning, message='invalid escape sequence')

# --- Setup Logger ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# --- Load Environment Variables ---
load_dotenv()

c:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πανελλήνιες\panellinies_exams_dataset\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
data_path = os.getenv("DATA_DIR")

if data_path:
    data_dir = Path(data_path)
    logger.info(f"Ξεκινάει η αναζήτηση στον φάκελο: {data_dir}")
    
    all_pairs = get_file_pairs (data_dir, target_school="GEL")
    logger.info(f"Βρέθηκαν συνολικά {len(all_pairs)} ζευγάρια αρχείων (JSON/MD).")
else:
    logger.error("Το DATA_DIR δεν βρέθηκε στο .env αρχείο!")

2026-04-20 16:23:12 - INFO - Ξεκινάει η αναζήτηση στον φάκελο: C:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πανελλήνιες\panellinies_exams_dataset\data
2026-04-20 16:23:12 - INFO - Βρέθηκαν συνολικά 59 ζευγάρια αρχείων (JSON/MD).


In [3]:
main_dataset = []

for pair in all_pairs:
    json_path = pair["json"]
    md_path = pair["md"]
    
    qa_list = merge_qa_data(json_path, md_path)
    
    main_dataset.extend(qa_list)

logger.info (f"Η ενοποίηση ολοκληρώθηκε! Βρέθηκαν συνολικά {len(main_dataset)} ερωτήσεις-απαντήσεις!")

2026-04-20 16:23:15 - INFO - Η ενοποίηση ολοκληρώθηκε! Βρέθηκαν συνολικά 1318 ερωτήσεις-απαντήσεις!


In [4]:
subject_translation = {
    "nea_ellinika": "greek_language",
    "arxaia": "ancient_greek",
    "istoria": "history",
    "latinika": "latin",
    "biologia": "biology",
    "fysiki": "physics",
    "ximeia": "chemistry",
    "pliroforiki": "computer_science",
    "arxes_oikonomikis_theorias": "economics",
    "mathimatika": "mathematics"
}

In [5]:
for item in main_dataset:
    q_text = item.get("question","")
    q_choices = item.get("choices",[])
    ans_text = item.get("answer","")
    images_list = item.get("images", [])
    marks = item.get("mark", [])
    
    form_type = detect_exercise_type(q_text,q_choices)
    item["format"] = form_type
    ans_idx = find_answer_index(q_choices,ans_text)
    item["answer_index"] = ans_idx
    item["reference"] = apply_reference_tag(item)
    
    old_subj = item.get("subject", "")
    new_subj = subject_translation.get(old_subj, old_subj)
    item["subject"] = new_subj
    
    #parsing image description and transcription
    all_descriptions = []
    all_transcriptions = []
    all_paths = []
    
    for img_dict in images_list:
        desc = img_dict.get("description","")
        if desc:
            all_descriptions.append(desc)
        transc = img_dict.get("transcription",[])
        if transc and isinstance(transc, list):
            joined_transc = ", ".join(transc)
            all_transcriptions.append(joined_transc)
        
        img_path = img_dict.get("path", "")
        if img_path:
            all_paths.append(img_path)
    
    mark_list = []
    
    for mark_text in marks:
        match = re.search(r'\d+\.?\d*', str(mark_text))
        if match:
            num_str = match.group()
            if "." in num_str:
                mark_list.append(float(num_str))
            else:
                mark_list.append(int(num_str))
    
    if len(mark_list) == 1:
        item["points"] = mark_list[0]
    elif len(mark_list) > 1:
        item["points"] = sum(mark_list)
    else:
        item["points"] = None
            
    item["image_description"] = " | ".join(all_descriptions)
    item["image_transcription"] = " | ".join(all_transcriptions)
    item["images"] = all_paths
    item.pop("mark", None)
    
    year = item.get("year", "")
    old_id = item.get("id", "")
    school_type = str(item.get("school_type", "gel")).lower()
    item["id"] = f"{new_subj}_{school_type}_{year}_{old_id}"

In [ ]:
images_found = 0
print("--- Ερωτήσεις που βρέθηκαν να έχουν εικόνες ---")

for item in main_dataset:
    imgs = item.get("images", [])
    
    if isinstance(imgs, list) and len(imgs) > 0:
        images_found += 1
        print(f"ID: {item.get('id')} στο μάθημα {item.get('subject')} ({item.get('year')}) - Περιέχει {len(imgs)} εικόνα/ες")

print(f"\nΣυνολικά βρέθηκαν {images_found} ερωτήσεις (IDs) με εικόνες.")

In [6]:
results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

output_file = results_dir / "panellinies_dataset.xlsx"

In [7]:
df = pd.DataFrame(main_dataset)
df = df.rename (columns={"answer": "answer_text"})
my_columns = [
    "id",
    "subject",
    "format",
    "reference",
    "question",
    "input",
    "images",
    "choices",
    "answer_text",
    "answer_index",
    "image_description",
    "image_transcription",
    "points",
    "year",
    "school_type"
]

df = df[my_columns]

In [8]:
df.to_excel(output_file, index=False)

logger.info (f"Tο αρχείο δημιουργήθηκε επιτυχώς στο: {output_file.resolve()}!")

df.head()

2026-04-20 16:23:32 - INFO - Tο αρχείο δημιουργήθηκε επιτυχώς στο: C:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πανελλήνιες\panellinies_exams_dataset\results\panellinies_dataset.xlsx!


,id,subject,format,reference,question,input,images,choices,answer_text,answer_index,image_description,image_transcription,points,year,school_type
0,ancient_greek_gel_2025_Α1.α.1,ancient_greek,multiple_choice,passage,Ποια είναι η κύρια αιτία η οποία εμποδίζει του...,Πλάτωνος Πολιτεία (514a-515c)\n\nΜετὰ ταῦτα δή...,[],"[α. Οι αλυσίδες στους αυχένες τους., β. Το σκο...",α,0.0,,,2.0,2025,GEL
1,ancient_greek_gel_2025_Α1.α.2,ancient_greek,multiple_choice,passage,Πού βρίσκεται το πυρ σε σχέση με τους δεσμώτες;,Πλάτωνος Πολιτεία (514a-515c)\n\nΜετὰ ταῦτα δή...,[],"[α. Μπροστά τους, χαμηλά., β. Επάνω και πίσω τ...",β,1.0,,,2.0,2025,GEL
2,ancient_greek_gel_2025_Α1.α.3,ancient_greek,multiple_choice,passage,"Τα «σκεύη», οι «ἀνδριάντες» και τα «ἄλλα ζῷα» ...",Πλάτωνος Πολιτεία (514a-515c)\n\nΜετὰ ταῦτα δή...,[],"[α. είναι πραγματικά ζώα της σπηλιάς., β. δημι...",β,1.0,,,2.0,2025,GEL
3,ancient_greek_gel_2025_Α1.β,ancient_greek,open_ended,passage,"«παρ’ ἣν», «ὑπὲρ ὧν»: Σε ποια λέξη του αρχαίου...",Πλάτωνος Πολιτεία (514a-515c)\n\nΜετὰ ταῦτα δή...,[],[],παρ ́ἥν: αναφέρεται στην ὁδόν\nὑπέρ ὧν: αναφέρ...,NaN,,,4.0,2025,GEL
4,ancient_greek_gel_2025_B1,ancient_greek,open_ended,passage,"Ποιος είναι ο βασικός εκφραστικός τρόπος, με τ...",Πλάτωνος Πολιτεία (514a-515c)\n\nΜετὰ ταῦτα δή...,[],[],Ο κυριότερος εκφραστικός τρόπος με τον οποίο ο...,NaN,,,10.0,2025,GEL


In [9]:
reference_file = Path("../results/panellinies_dataset.xlsx")

compare_results = compare(df, reference_file)

if isinstance (compare_results, dict):
    print("\n--- IDs που υπάρχουν ΜΟΝΟ στο νέο dataset ---")
    display(compare_results["only_current"][["id"]].head(20))
    
    print("\n--- IDs που υπάρχουν ΜΟΝΟ στο παλιό dataset ---")
    display(compare_results["only_reference"][["id"]].head(20))

2026-04-20 16:23:36 - INFO - 🔍 Σύγκριση με το αρχείο: ..\results\panellinies_dataset.xlsx
2026-04-20 16:23:37 - INFO - 📊 Στατιστικά σύγκρισης:
2026-04-20 16:23:37 - INFO -    - Match (ίδια ids και στα δύο): 1318
2026-04-20 16:23:37 - INFO -    - Only in Current: 0
2026-04-20 16:23:37 - INFO -    - Only in Reference: 0
2026-04-20 16:23:37 - INFO - ✅ Η στήλη 'subject' ταιριάζει πλήρως.
2026-04-20 16:23:37 - INFO - ✅ Η στήλη 'format' ταιριάζει πλήρως.
2026-04-20 16:23:37 - INFO - ✅ Η στήλη 'reference' ταιριάζει πλήρως.
2026-04-20 16:23:37 - INFO - ✅ Η στήλη 'question' ταιριάζει πλήρως.
2026-04-20 16:23:37 - INFO - ✅ Η στήλη 'input' ταιριάζει πλήρως.
2026-04-20 16:23:37 - INFO - ✅ Η στήλη 'choices' ταιριάζει πλήρως.
2026-04-20 16:23:37 - INFO - ✅ Η στήλη 'answer_text' ταιριάζει πλήρως.
2026-04-20 16:23:37 - INFO - ✅ Η στήλη 'answer_index' ταιριάζει πλήρως.
2026-04-20 16:23:37 - INFO - ✅ Η στήλη 'image_description' ταιριάζει πλήρως.
2026-04-20 16:23:37 - INFO - ✅ Η στήλη 'image_transcriptio


--- IDs που υπάρχουν ΜΟΝΟ στο νέο dataset ---


,id



--- IDs που υπάρχουν ΜΟΝΟ στο παλιό dataset ---


,id


In [10]:
#testing conversion of matching questions into multiple choices ones for LLM evaluation
test_df = apply_matching_processing(df)

matching_only_df = test_df[test_df['format'] == 'matching'][['id', 'processed_choices', 'new_answer_index']]

show(matching_only_df,
     layout={"top1": "searchBuilder"},
     buttons=[
         "pageLength",
         {"extend": "excelHtml5", "title": "test_df_matching_questions"}
     ]
)

Loading ITables v2.7.3 from the internet... (need help?)


In [12]:
#matching_questions validation
first_matching = test_df[test_df['format'] == 'matching'].iloc[0]

print(f"Πλήθος επιλογών: {len(first_matching['processed_choices'])}")
print(f"Η 1η επιλογή είναι: {first_matching['processed_choices'][0]}")

Πλήθος επιλογών: 4
Η 1η επιλογή είναι: [1-β, 2-α, 3-γ, 4-β, 5-α, 6-γ, 7-γ]
